# 06 – Modellinterpretation, kritische Diskussion & Ethik

Dieses Notebook beantwortet Teilfrage 1 (wichtigste Merkmale) und Teilfrage 5
(ethische/methodische Grenzen).

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import config, utils, preprocessing as prep, train_models as tm, interpretation as itp
feat = utils.load_processed("feature_matrix")
X_train, X_test, y_train, y_test = prep.make_train_test_split(feat)
fitted = tm.fit_all_models(X_train, y_train)

22:00:34 | INFO    | Split: Train=246008 (Positivrate 0.081), Test=61503 (Positivrate 0.081)
22:00:34 | INFO    | Feature-Typen: 143 numerisch, 16 kategorial.
22:00:42 | INFO    | Gefittet: dummy
22:01:08 | INFO    | Gefittet: logreg
22:01:43 | INFO    | Gefittet: random_forest
c:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] Das System kann die angegebene Datei nicht finden
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^

## 1. Permutation Importance (modellagnostisch)

**Warum bevorzugt:** robuster als impurity-based Importance und auf dem Testset
interpretierbar. **Grenze:** bei korrelierten Merkmalen kann Wichtigkeit "geteilt"
werden. **Kausalität:** Wichtigkeit ist ASSOZIATIV, nicht kausal.

In [2]:
model = fitted["random_forest"]
pi = itp.permutation_importances(model, X_test, y_test, n_repeats=10)
plt.show()
pi.head(15).round(4)

22:24:29 | INFO    | Tabelle gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\tables\interpretation\permutation_importance.csv
22:24:30 | INFO    | Abbildung gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\figures\interpretation\permutation_importance.png
22:24:30 | INFO    | Permutation Importance berechnet. Top-3: ['external_score_mean', 'external_score_min', 'external_score_max']
C:\Users\annis\AppData\Local\Temp\ipykernel_27916\2969259280.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,feature,importance_mean,importance_std
0,external_score_mean,0.0246,0.0021
1,external_score_min,0.0177,0.0019
2,external_score_max,0.0095,0.0011
3,EXT_SOURCE_3,0.0087,0.0008
4,EXT_SOURCE_2,0.0085,0.0008
5,annuity_credit_ratio,0.0038,0.0009
6,NAME_EDUCATION_TYPE,0.0031,0.0006
7,goods_credit_ratio,0.0030,0.0009
8,age_years,0.0030,0.0006
9,AMT_ANNUITY,0.0024,0.0008


In [3]:
print(itp.plausibility_check(pi))

Top-5 Merkmale: ['external_score_mean', 'external_score_min', 'external_score_max', 'EXT_SOURCE_3', 'EXT_SOURCE_2']. Keine als sensibel markierten Merkmale unter den Top-5.


## 2. Impurity-based Importance (ergänzend, mit Vorbehalt)

In [ ]:
ti = itp.tree_feature_importance(model, top=15)
ti.head(15).round(4) if ti is not None else "Modell ohne impurity-based Importance." 

22:24:30 | INFO    | Tabelle gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\tables\interpretation\tree_feature_importance.csv


,feature,importance
0,num__external_score_mean,0.0475
1,num__external_score_min,0.0391
2,num__external_score_max,0.0343
3,num__EXT_SOURCE_2,0.0292
4,num__EXT_SOURCE_3,0.0271
5,num__annuity_credit_ratio,0.0173
6,num__DAYS_EMPLOYED,0.0171
7,num__external_score_std,0.0168
8,num__DAYS_BIRTH,0.0167
9,num__age_years,0.0161


: 

## 3. (Optional) SHAP

Falls `shap` installiert ist, wird ein Summary-Plot erzeugt; sonst wird der Schritt
übersprungen (das Projekt bleibt vollständig lauffähig).

In [ ]:
_ = itp.try_shap_summary(model, X_test.iloc[:200])

c:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 4. Sind die wichtigen Merkmale fachlich plausibel?

Typischerweise dominieren **externe Bonitätsscores** und **Verschuldungs-/Zahlungs-
kennzahlen** – das ist fachlich plausibel. Ein Warnsignal wäre, wenn IDs oder offensichtliche
Proxys für sensible Merkmale dominierten. **Wichtig:** Plausibilität ≠ Kausalität.

## 5. Kritische Diskussion & Ethik

**Datenqualität & fehlende Werte:** Viele Merkmale fehlen teils stark; der Fehl-
Mechanismus (MCAR/MAR/MNAR) ist unbekannt. Imputation kann Verzerrungen einführen.

**Historische Verzerrungen (Bias):** Das Modell lernt aus VERGANGENEN Entscheidungen.
Waren diese diskriminierend, reproduziert/verstärkt das Modell die Diskriminierung
("Bias in, bias out").

**Proxy-Variablen für sensible Merkmale:** Selbst ohne direkte Nutzung von Geschlecht/
Herkunft können Merkmale wie Wohnregion, Familienstand oder Beruf als Proxys wirken und
zu **indirekter Diskriminierung** führen. Wir haben solche Merkmale markiert
(`config.SENSITIVE_OR_PROXY_FEATURES`) und ihre Wichtigkeit geprüft.

**Vorhersage ≠ Entscheidung:** Das Modell schätzt eine Wahrscheinlichkeit. Die
Kreditentscheidung ist ein separater, wertbehafteter Schritt (Schwelle, Härtefälle,
menschliche Prüfung).

**Interpretierbarkeit & Recht:** Automatisierte Entscheidungen mit erheblichen Wirkungen
unterliegen in der EU besonderen Anforderungen (DSGVO Art. 22; Erwägungsgrund 71 nennt
u. a. das Recht auf Erläuterung – die genaue Reichweite ist juristisch umstritten,
**zu prüfen**).

**Fairness & Transparenz:** Fairness ist mehrdeutig (es gibt mehrere, teils
unvereinbare Fairness-Definitionen; Kleinberg et al., 2016, **zu prüfen**). Ein
Fairness-Audit (z. B. Gruppen-Recall) ist vor produktivem Einsatz nötig.

**Datenschutz:** Auch anonymisierte Finanzdaten sind sensibel; Zweckbindung und
Datensparsamkeit (DSGVO) sind zu beachten.

**Klare Einordnung:**
> Dieses Modell ist ein **akademisches Analyseprojekt** und darf **nicht** als alleinige
> Grundlage für reale Kreditentscheidungen dienen. Reale Entscheidungen erfordern
> menschliche Prüfung, regulatorische Konformität und ein Fairness-/Robustheits-Audit.

## 6. Limitationen dieses Projekts (ehrlich)

- **Demo-Daten:** Standardmäßig synthetisch -> Kennzahlen nicht übertragbar.
- **Aggregations-Trade-off:** zeitliche Dynamik der Nebentabellen geht teils verloren.
- **Kein zeitlicher Split:** ein realistischer Kreditscore sollte zeitlich validiert
  werden (Training auf älteren, Test auf neueren Anträgen), um "Lernen aus der Zukunft"
  auszuschließen. Hier vereinfachter zufälliger Split.
- **Begrenztes Tuning:** kleiner Suchraum aus Effizienzgründen.

---
### Quellen (Auswahl; im README/Theorieteil vollständig, Unsicheres als 'zu prüfen')
- Lundberg, S. M., & Lee, S.-I. (2017). A unified approach to interpreting model
  predictions (SHAP). *NeurIPS.*
- Molnar, C. (2022). *Interpretable Machine Learning* (2nd ed.).
- Barocas, S., Hardt, M., & Narayanan, A. (2019). *Fairness and Machine Learning.*